In [3]:
!unzip -q data.zip

replace data/labels.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [4]:
!pip install transformers torch pillow pandas

import torch
from torch.utils.data import Dataset
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        image = Image.open(row["image"]).convert("RGB")

        encoding = self.processor(image, return_tensors="pt")
        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=128,
            truncation=True
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [11]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

dataset = UrduOCRDataset("data/labels.csv", processor)

sample = dataset[0]

print("Pixel Values Shape:", sample["pixel_values"].shape)
print("Labels Shape:", sample["labels"].shape)

print("Dataset is working correctly!")

Dataset loaded: 180 samples
Pixel Values Shape: torch.Size([3, 384, 384])
Labels Shape: torch.Size([128])
Dataset is working correctly!


In [12]:
# Train/Test Split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

print(f"Training Samples: {train_size}")
print(f"Testing Samples: {test_size}")

Training Samples: 144
Testing Samples: 36
